# NEXORA 2026 — Notebook 05: Historical Backtesting Engine (Phase 6.2)

**Target Contract:** `reports/PHASE_6_1_BACKTEST_TARGET.md` (`requested_on` primary attribution)
**Feature Library:** `reports/PHASE_5_2_FEATURE_EXTRACTION.md`, `src/nexora/feature_extractor.py`
**Backtesting Package:** `src/nexora/backtesting/`
**Objective:** Leakage-safe backtesting across all 26 historical Mondays (2025-08-04 to 2026-01-26).
Compares deterministic candidate ranking strategies A–F against the 3-sigma control baseline.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from src.nexora.data_loader import DataLoader, normalize_gateway_id
from src.nexora.feature_extractor import FeatureExtractor
from src.nexora.target_constructor import TargetConstructor
from src.nexora.backtesting import (
    HistoricalBacktester,
    ALL_STRATEGIES,
    run_all_leakage_tests,
)

print('All modules successfully loaded!')


All modules successfully loaded!


## 1. Automated Anti-Leakage & Determinism Test Suite

Before evaluating any strategies, we execute our automated verification suite to prove:
1. **Synthetic Telemetry Isolation:** Large future events at $t \ge T$ do not alter features (primary feature leakage test).
2. **Engineer Review Temporal Guard:** Review labels dated `2026-02-15` are strictly unavailable for $T \le \text{2026-02-15}$, and all post-review records satisfy $reviewed\_on < as\_of\_date$.
3. **Meter Data Temporal Boundary:** Historical weeks exclude future meter reads ($week\_start\_dt < T$).
4. **Ranking Determinism:** Two consecutive backtest runs produce identical rankings, scores, and metrics.

> **Field-Visit Structural Isolation:** Under the NEXORA architecture, `field_visits.csv` is loaded exclusively by `TargetConstructor` to build evaluation targets. `FeatureExtractor` never reads, references, or imports `field_visits.csv`. Therefore, field-visit leakage into feature extraction is structurally impossible by modular code separation.


In [2]:
leakage_results = run_all_leakage_tests()
print('Leakage and Determinism Test Results:')
for test_name, passed in leakage_results.items():
    status = 'PASSED [OK]' if passed else 'FAILED [X]'
    print(f'  - {test_name}: {status}')
assert all(leakage_results.values()), 'One or more leakage tests failed!'


Leakage and Determinism Test Results:
  - synthetic_telemetry_isolation: PASSED [OK]
  - engineer_review_temporal_guard: PASSED [OK]
  - meter_data_temporal_boundary: PASSED [OK]
  - ranking_determinism: PASSED [OK]


## 2. Full 26-Week Historical Backtest Simulation

We instantiate `HistoricalBacktester` and execute across all 26 historical Mondays:
- Horizon: 2025-08-04 to 2026-01-26
- Strategies: Baseline (3-Sigma), Candidate A (Core), Candidate B (Severity), Candidate C (Severe Offline), Candidate D (Long-Term), Candidate E (Reliability), Candidate F (Silence Override)
- Target Attribution: `requested_on \in [T, T+7\text{d})` with delayed outcome realization tracking


In [3]:
backtester = HistoricalBacktester()
weekly_df, summary_df = backtester.run()
saved_paths = backtester.save_results()

print(f'\nBacktest simulation finished successfully!')
print(f'Total weekly evaluation rows: {len(weekly_df)}')


Starting historical backtest across 26 weeks for 7 strategies...
  [01/26] Evaluating decision Monday 2025-08-04...


  [02/26] Evaluating decision Monday 2025-08-11...


  [03/26] Evaluating decision Monday 2025-08-18...


  [04/26] Evaluating decision Monday 2025-08-25...


  [05/26] Evaluating decision Monday 2025-09-01...


...

  [23/26] Evaluating decision Monday 2026-01-05...


  [24/26] Evaluating decision Monday 2026-01-12...


  [25/26] Evaluating decision Monday 2026-01-19...


  [26/26] Evaluating decision Monday 2026-01-26...


Successfully saved backtest artifacts to D:\lpdg-nexora-2026\reports\backtest:
  - weekly_results: backtest_weekly_results.csv
  - summary: backtest_summary.csv
  - top_k: backtest_topk.csv
  - rankings: backtest_rankings.csv

Backtest simulation finished successfully!
Total weekly evaluation rows: 182


## 3. Macro Strategy Comparison

Aggregated results across all 26 historical weeks (390 total selections per strategy, 116 available repairs):


In [4]:
display_cols = [
    'strategy',
    'total_repairs_captured',
    'total_repairs_available',
    'overall_repair_capture',
    'overall_repair_yield',
    'overall_false_alarm_rate',
    'overall_precision_like_prop',
    'total_false_alarms',
    'total_missed_repairs',
    'total_fa_cost_proxy_eur',
    'total_missed_repair_cost_proxy_eur',
]
summary_display = summary_df[display_cols].copy()
summary_display['overall_repair_capture'] = summary_display['overall_repair_capture'].apply(lambda x: f'{x*100:.2f}%')
summary_display['overall_repair_yield'] = summary_display['overall_repair_yield'].apply(lambda x: f'{x*100:.2f}%')
summary_display['overall_false_alarm_rate'] = summary_display['overall_false_alarm_rate'].apply(lambda x: f'{x*100:.2f}%')
summary_display['overall_precision_like_prop'] = summary_display['overall_precision_like_prop'].apply(lambda x: f'{x*100:.2f}%')
summary_display['total_fa_cost_proxy_eur'] = summary_display['total_fa_cost_proxy_eur'].apply(lambda x: f'EUR {x:,.0f}')
summary_display['total_missed_repair_cost_proxy_eur'] = summary_display['total_missed_repair_cost_proxy_eur'].apply(lambda x: f'EUR {x:,.0f}')

print(summary_display.to_string(index=False))


                   strategy  total_repairs_captured  total_repairs_available overall_repair_capture overall_repair_yield overall_false_alarm_rate overall_precision_like_prop  total_false_alarms  total_missed_repairs total_fa_cost_proxy_eur total_missed_repair_cost_proxy_eur
  Candidate_C_SevereOffline                      42                      116                 36.21%               10.77%                    5.64%                      65.62%                  22                    74               EUR 8,360                         EUR 44,400
            Baseline_3Sigma                      41                      116                 35.34%               10.51%                    3.59%                      74.55%                  14                    75               EUR 5,320                         EUR 45,000
           Candidate_A_Core                      40                      116                 34.48%               10.26%                    5.64%                      64.52%  

## 4. Top-K Cumulative Repair Capture Curves ($K = 1 \dots 15$)

To understand whether signal is concentrated near the very top of the ranking or distributed across ranks 1..15, we evaluate cumulative repair capture at each cutoff $K$:


In [5]:
topk_df = pd.read_csv('reports/backtest/backtest_topk.csv')
piv = topk_df.pivot(index='k', columns='strategy', values='total_repairs_captured')
print('Cumulative Repairs Captured by Rank Cutoff K (Total Available = 116):')
print(piv.to_string())


Cumulative Repairs Captured by Rank Cutoff K (Total Available = 116):
strategy  Baseline_3Sigma  Candidate_A_Core  Candidate_B_Severity  Candidate_C_SevereOffline  Candidate_D_LongTerm  Candidate_E_Reliability  Candidate_F_SilenceOverride
k                                                                                                                                                                       
1                       6                 0                     1                          3                     1                        2                            1
2                      13                 5                     3                          5                     4                        4                            3
3                      16                10                     7                         12                     8                       10                            6
4                      17                10                     8                    

## 5. Weekly Metric Distribution & Stability

We examine the distribution of weekly repair yield and false alarm rate across the 26 historical weeks:


In [6]:
stab_cols = [
    'strategy',
    'weekly_capture_mean',
    'weekly_capture_std',
    'weekly_capture_min',
    'weekly_capture_max',
    'weekly_yield_mean',
    'weekly_yield_std',
    'weekly_yield_min',
    'weekly_yield_max',
    'weekly_fa_rate_mean',
    'weekly_fa_rate_std',
]
print(summary_df[stab_cols].to_string(index=False))


                   strategy  weekly_capture_mean  weekly_capture_std  weekly_capture_min  weekly_capture_max  weekly_yield_mean  weekly_yield_std  weekly_yield_min  weekly_yield_max  weekly_fa_rate_mean  weekly_fa_rate_std
  Candidate_C_SevereOffline             0.393727            0.288572                 0.0                 1.0           0.107692          0.080171               0.0          0.266667             0.056410            0.055593
            Baseline_3Sigma             0.373031            0.273140                 0.0                 1.0           0.105128          0.078141               0.0          0.266667             0.035897            0.063191
           Candidate_A_Core             0.393727            0.279774                 0.0                 1.0           0.102564          0.071132               0.0          0.266667             0.056410            0.052298
       Candidate_B_Severity             0.351099            0.275272                 0.0                 1.0

## 6. Key Operational Insights

1. **Candidate C (Severe Offline) achieves highest cumulative repair capture:** 42 confirmed repairs captured (36.21%), edging out Baseline (41) and Candidate A (40).
2. **Baseline 3-Sigma has highest precision in the top ranks (K=1..5):** 3-sigma flags capture 19 repairs in Top-5 vs Candidate C's 14, and incurs fewer false alarms (14 vs 22).
3. **Candidate A (Persistence/Core) is robust:** Using only F02, F09, and F16 captures 40 repairs without requiring severity transformations.
4. **Silence Override (Candidate F):** Completely silent gateways ($F05=1$) are relatively rare in the active universe; prioritizing them alone does not boost observed dispatch capture due to historical selection bias (legacy dispatchers often did not inspect completely silent gateways unless complaints occurred).
5. **High Proportion of Unobserved Gateways in Top-15:** For all strategies, ~83–85% of Top-15 selections were not visited historically (`UNOBSERVED`). Because `UNOBSERVED != healthy`, these gateways represent latent candidates that legacy dispatchers never inspected.
